# CB Upfront Costs Innovation Analysis

## 📋 Overview

**Purpose**: Analyzes companies developing innovations to lower upfront costs for heat pump deployment from Crunchbase data using LLM-validated filtering to identify relevant companies in the upfront costs reduction space.

**Key Features**:
- LLM validation of company relevance to upfront costs innovation
- Comprehensive investment analysis with funding stages (early/growth/late/other)
- Geographic distribution analysis with EU/UK/Other grouping
- Enhanced funding stages analysis (grouped investment lifecycle)
- Landscape visualization with clustering analysis
- Automated Google Sheets export with separated main data and chart data

**Focus Areas**:
- Technologies to speed up installation processes
- Standardized/modular systems for economies of scale  
- Manufacturing cost reduction and mass production solutions
- Innovative financing mechanisms and payment structures
- Supply chain optimization and automation tools

**Expected Results**: 
- Upfront Costs Innovation Companies: ~20-40 companies
- Focus on cost reduction, installation efficiency, and financing innovation
- Analysis of commercial investment patterns in heat pump cost reduction

---

In [2]:
from discovery_utils.getters import gtr
from discovery_utils.getters import crunchbase
from discovery_utils.utils import search

from discovery_utils.utils.llm.batch_check import LLMProcessor, generate_relevance_check_system_message

from discovery_mission_radar import PROJECT_DIR
from discovery_utils.utils.io import remap_dict

In [3]:
OUTPUT_DIR = PROJECT_DIR / 'data/upfront_costs_innovation/cb/'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DB_DIR = PROJECT_DIR / 'tmp/vector_db/'
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
SCORE_THRESHOLD = 0.3

# Single config approach - simplified directory structure
primary_config = "upfront_costs"

# Create simplified directory structure directly under cb/
raw_dir = OUTPUT_DIR / "raw"
filtered_dir = OUTPUT_DIR / "filtered"
produce_stats_dir = OUTPUT_DIR / "produce_stats"
charts_dir = OUTPUT_DIR / "charts"
chart_csvs_dir = OUTPUT_DIR / "chart_csvs"

# Create all directories
for directory in [raw_dir, filtered_dir, produce_stats_dir, charts_dir, chart_csvs_dir]:
    directory.mkdir(exist_ok=True)

In [4]:
CB = crunchbase.CrunchbaseGetter(vector_db_path=VECTOR_DB_DIR)

# Configuration flags for funding analysis
INCLUDE_LATE_STAGE = True  # Set to True to include late-stage funding, False for early-stage only
INCLUDE_GROWTH_STAGE = True  # Set to True to include growth-stage funding
INCLUDE_OTHER_STAGE = True  # Set to True to include other funding stages

investment_stages = {
    'early_stage': ['pre_seed', 'seed', 'angel', 'series_a', 'series_b', 'convertible_note',
                   'equity_crowdfunding', 'product_crowdfunding', 'non_equity_assistance', 'initial_coin_offering'],
    'growth_stage': ['series_c', 'series_d', 'series_e', 'series_f', 'series_g', 'series_h', 'series_i', 'series_j'],
    'late_stage': ['private_equity', 'post_ipo_equity', 'post_ipo_debt', 'post_ipo_secondary', 'secondary_market'],
    'other': ['corporate_round', 'debt_financing', 'grant', 'series_unknown', 'undisclosed']
}

# Build funding round types based on configuration
FUNDING_ROUND_TYPES = investment_stages['early_stage'].copy()
if INCLUDE_GROWTH_STAGE:
    FUNDING_ROUND_TYPES.extend(investment_stages['growth_stage'])
if INCLUDE_LATE_STAGE:
    FUNDING_ROUND_TYPES.extend(investment_stages['late_stage'])
if INCLUDE_OTHER_STAGE:
    FUNDING_ROUND_TYPES.extend(investment_stages['other'])

# Create descriptive string for analysis
stage_descriptions = ['Early-stage']
if INCLUDE_GROWTH_STAGE:
    stage_descriptions.append('Growth-stage') 
if INCLUDE_LATE_STAGE:
    stage_descriptions.append('Late-stage')
if INCLUDE_OTHER_STAGE:
    stage_descriptions.append('Other')
FUNDING_ROUND_TYPES_STR = ', '.join(stage_descriptions) + ' funding'

print(f"Analysis will include: {FUNDING_ROUND_TYPES_STR}")
print(f"Funding round types: {len(FUNDING_ROUND_TYPES)} types")

investment_type_to_stage = remap_dict(investment_stages)

2025-07-30 16:56:11,547 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-07-30 16:56:11,709 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2025-07-28
Analysis will include: Early-stage, Growth-stage, Late-stage, Other funding
Funding round types: 28 types


In [5]:
import pandas as pd

# Check if processed data already exists
upfront_csv_path = filtered_dir / f"relevant_{primary_config}_llm_filtered.csv"

if upfront_csv_path.exists():
    print(f"✓ Found existing {primary_config} filtered data. Loading from file...")
    df_filtered = pd.read_csv(upfront_csv_path)
    print(f"Loaded {len(df_filtered)} {primary_config} companies")
else:
    print(f"⚠️  No existing {primary_config} data found. Running full filtering process...")
    
    # Define single config
    config = primary_config
    config_file = f"config_{config}.yaml"

    # Run keyword+vector search
    Search = search.SearchDataset(CB, CB.organisations_enriched, config_file)
    df = (
        Search.do_search()
               .merge(CB.descriptions[['id','description']], on='id', how='left')
               .fillna({'description':'','short_description':'','name':''})
               .assign(text=lambda d: d['name'] + '. ' + d['short_description'] + ' ' + d['description'])
    )
    relevant = df.query(f"_score_avg > {SCORE_THRESHOLD}")
    csv_path_temp = raw_dir / f"relevant_{primary_config}.csv"
    jsonl_path = raw_dir / f"llm_check_MS_{primary_config}.jsonl"
    relevant.to_csv(csv_path_temp, index=False)

    print(f"Found {len(relevant)} companies above threshold {SCORE_THRESHOLD}")
    
    # LLM-based relevance check
    sys_msg = generate_relevance_check_system_message(config_file)
    processor = LLMProcessor(
        output_path=str(jsonl_path),
        system_message=sys_msg,
        session_name="mission_studio",
        output_fields=[{"name":"is_relevant","type":"str","description":"yes or no"}]
    )
    await processor.run(dict(zip(relevant['id'], relevant['text'])), batch_size=15, sleep_time=0.5)
    
    # Merge and filter
    df_csv = pd.read_csv(csv_path_temp)
    df_jsonl = pd.read_json(jsonl_path, lines=True)
    merged = df_csv.merge(df_jsonl[['id', 'is_relevant']], on='id', how='left')
    filtered = merged[merged['is_relevant'].str.lower() == 'yes']
    filtered.to_csv(filtered_dir / f"relevant_{primary_config}_llm_filtered.csv", index=False)
    rejected = merged[merged['is_relevant'].str.lower() == 'no']
    rejected.to_csv(filtered_dir / f"relevant_{primary_config}_rejected.csv", index=False)
    print(f"Filtered {len(filtered)} relevant entries out of {len(df_csv)} total.")
    print(f"Filtered {len(rejected)} rejected entries out of {len(df_csv)} total.")
    
    # Load the LLM-filtered CSV
    df_filtered = filtered.copy()

print(f"\n🎯 Data Summary:")
print(f"Upfront Costs Innovation: {len(df_filtered)} companies")

⚠️  No existing upfront_costs data found. Running full filtering process...
2025-07-30 16:56:16,833 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/organizations_full.parquet
2025-07-30 16:57:09,785 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: data/crunchbase/enriched/organizations_full.parquet
2025-07-30 16:57:09,818 - root - INFO - Folder /Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/tmp/vector_db/crunchbase-lancedb/ already exists. Set overwrite=True to download again.
2025-07-30 16:57:09,825 - root - INFO - Connected with database crunchbase-lancedb. Available tables: ['company_embeddings']
2025-07-30 16:57:09,892 - root - ERROR - Error creating FTS index: Index already exists. Use replace=True to overwrite.
2025-07-30 16:57:21,941 - discovery_utils - INFO - Keyword search found 0 matches
2025-07-30 16:57:21,980 - sentence_transformers.SentenceTransf

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.58it/s]


2025-07-30 16:57:49,571 - discovery_utils - INFO - Vector search found 21756 matches
2025-07-30 16:57:50,040 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/Crunchbase_2025-07-28/organization_descriptions.parquet
2025-07-30 16:58:10,786 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: data/crunchbase/Crunchbase_2025-07-28/organization_descriptions.parquet
Found 526 companies above threshold 0.3
2025-07-30 16:58:12,284 - root - INFO - Using OpenAI
2025-07-30 16:58:12,547 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
2025-07-30 16:58:12,592 - root - INFO - Processing batch 1/36
2025-07-30 16:58:15,889 - root - INFO - Processing batch 2/36
2025-07-30 16:58:19,982 - root - INFO - Processing batch 3/36
2025-07-30 16:58:22,234 - 

In [6]:
from discovery_utils.utils import (
    analysis_crunchbase,
    analysis,
    charts,
    viz_landscape,
)
import altair as alt

# FUNDING_ROUND_TYPES and FUNDING_ROUND_TYPES_STR are now defined in the configuration cell above
print(f"Using funding stages: {FUNDING_ROUND_TYPES_STR}")
print(f"Total funding round types: {len(FUNDING_ROUND_TYPES)}")

# Column definitions for export
cols_funding_rounds = [
    "funding_round_name", 
    "org_name",
    "theme",
    "cb_url",
    "country_code", 
    "region_nesta",
    "region",
    "city",
    "year",
    "announced_on",
    "investment_type", 
    "investment_stage",
    "raised_amount_gbp",
    "raised_amount_usd",
    "raised_amount", 
    "raised_amount_currency_code",
    "post_money_valuation_usd", 
    "post_money_valuation",
    "post_money_valuation_currency_code", 
    "investor_name",
]

cols_companies = [
    "name", 
    "short_description", 
    "founded_on", 
    'created_at',    
    "cb_url", 
    "homepage_url", 
    "ai_relevance_check",
    "theme",
    'landscape_category',
    'landscape_keyword_cluster',
    'mission_labels',
    'topic_labels',
    "rank", 
    "country_code", 
    'region_nesta',
    "region", 
    "city", 
    "status", 
    "category_list", 
    "closed_on", 
    "employee_count", 
    "email", 
    "phone", 
    "facebook_url", 
    "linkedin_url", 
    "twitter_url", 
    "logo_url", 
    'recent_funding',    
    "num_exits", 
    "num_funding_rounds", 
    "last_funding_on", 
    "investment_funding_gbp", 
    "num_investment_rounds", 
    "grant_funding_gbp", 
    "num_grants", 
    "total_funding_gbp", 
    "smart_money", 
    "_score_keywords",
    "_score_vectors",
    "_score_avg",
]

def aggregate_by_funding_stages(funding_rounds_df):
    """
    Aggregate funding rounds by funding stage (early/growth/late/other) instead of individual investment types.
    """
    # Add investment stage column using existing mapping
    funding_rounds_with_stage = funding_rounds_df.copy()
    funding_rounds_with_stage['investment_stage'] = funding_rounds_with_stage['investment_type'].map(investment_type_to_stage)
    
    # Group by year and investment stage
    aggregated = (
        funding_rounds_with_stage
        .groupby(['year', 'investment_stage'])
        .agg({
            'raised_amount_gbp': 'sum',
            'funding_round_id': 'count'
        })
        .reset_index()
        .rename(columns={
            'funding_round_id': 'num_funding_rounds',
            'raised_amount_gbp': 'total_amount'
        })
    )
    
    # Convert to millions for easier reading
    aggregated['total_amount_millions'] = aggregated['total_amount']
    
    return aggregated

def chart_funding_stages(aggregated_funding_stages_df, chart_title=None):
    """
    Chart funding amounts by stage (early/growth/late/other) over time.
    """
    chart = alt.Chart(aggregated_funding_stages_df).mark_bar().encode(
        x=alt.X('year:O', title='Year'),
        y=alt.Y('total_amount_millions:Q', title='Total Amount (£ billions)'),
        color=alt.Color('investment_stage:N', title='Funding Stage'),
        tooltip=['year:O', 'investment_stage:N', 'total_amount_millions:Q', 'num_funding_rounds:Q']
    ).properties(
        width=600,
        height=400,
        title=chart_title or 'Funding Amounts by Stage Over Time'
    )
    
    return chart

def chart_funding_stages_counts(aggregated_funding_stages_df, chart_title=None):
    """
    Chart number of funding rounds by stage (early/growth/late/other) over time.
    """
    chart = alt.Chart(aggregated_funding_stages_df).mark_bar().encode(
        x=alt.X('year:O', title='Year'),
        y=alt.Y('num_funding_rounds:Q', title='Number of Funding Rounds'),
        color=alt.Color('investment_stage:N', title='Funding Stage'),
        tooltip=['year:O', 'investment_stage:N', 'num_funding_rounds:Q', 'total_amount_millions:Q']
    ).properties(
        width=600,
        height=400,
        title=chart_title or 'Number of Funding Rounds by Stage Over Time'
    )
    
    return chart

Using funding stages: Early-stage, Growth-stage, Late-stage, Other funding
Total funding round types: 28


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/aidan.kelly/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [8]:
from pathlib import Path

def produce_stats(CB, matching_ids: list[str], category_name: str, output_dir: Path) -> tuple:
    """ Produce stats for the companies and output charts """
    
    # Use the pre-created directories from the main structure
    charts_dir = output_dir / "charts"
    chart_csvs_dir = output_dir / "chart_csvs" 
    produce_stats_dir = output_dir / "produce_stats"
    
    # Ensure directories exist
    charts_dir.mkdir(parents=True, exist_ok=True)
    chart_csvs_dir.mkdir(parents=True, exist_ok=True)
    produce_stats_dir.mkdir(parents=True, exist_ok=True)
    
    # Check companies by querying ids
    matchings_orgs_df = CB.organisations_enriched.query("id in @matching_ids")

    # Get the funding rounds for the matching companies
    funding_rounds_df = (
        CB.select_funding_rounds(org_ids=matching_ids, funding_round_types=FUNDING_ROUND_TYPES)
    )

    # Organise investors by each funding round
    investors_df = (
        CB.funding_rounds_enriched
        .query("funding_round_id in @funding_rounds_df.funding_round_id")
        .groupby("funding_round_id")
        .agg(investor_name=("investor_name", list))
        .reset_index()
    )

    funding_rounds_df = (
        funding_rounds_df
        .drop(columns=["investor_name"])
        .merge(investors_df, on="funding_round_id", how="left").drop_duplicates("funding_round_id", keep="first")
    )

    # Generate basic time series
    ts_df = analysis_crunchbase.get_timeseries(
        matchings_orgs_df, 
        funding_rounds_df, 
        period='year', 
        min_year=2014, 
        max_year=2025
    )
    
    # Fill in missing years with zeros for better chart visualization
    ts_df_for_imputation = ts_df.copy()
    ts_df_for_imputation['year_datetime'] = pd.to_datetime(ts_df_for_imputation['year'], format='%Y')
    
    # Apply imputation using the datetime column
    ts_df_imputed = analysis.impute_empty_periods(
        ts_df_for_imputation.drop('year', axis=1), 
        time_period_col='year_datetime', 
        period='Y',  # 'Y' for annual/yearly data
        min_year=2014, 
        max_year=2025
    )
    
    # Convert back to integer year for downstream functions
    ts_df = ts_df_imputed.copy()
    ts_df['year'] = ts_df['year_datetime'].dt.year
    ts_df = ts_df.drop('year_datetime', axis=1)
    
    growth_rates = analysis.smoothed_growth(ts_df, year_start=2020, year_end=2024)
    growth_rates_df = pd.DataFrame(
        growth_rates, columns=[category_name]
    ).T.reset_index().rename(columns={'index': 'theme'})  

    # Breakdown of deal types (detailed)
    aggregated_funding_types_df = analysis_crunchbase.aggregate_by_funding_round_types(funding_rounds_df)
    
    # Breakdown by funding stages (grouped)
    aggregated_funding_stages_df = aggregate_by_funding_stages(funding_rounds_df)
    
    # Fill in missing years for funding stages data too
    imputed_stages_list = []
    for stage in aggregated_funding_stages_df['investment_stage'].unique():
        stage_data = aggregated_funding_stages_df[aggregated_funding_stages_df['investment_stage'] == stage].copy()
        
        # Convert year to datetime for imputation
        stage_data['year_datetime'] = pd.to_datetime(stage_data['year'], format='%Y')
        stage_data_for_imputation = stage_data.drop('year', axis=1)
        
        # Apply imputation
        stage_imputed = analysis.impute_empty_periods(
            stage_data_for_imputation,
            time_period_col='year_datetime',
            period='Y',
            min_year=2014,
            max_year=2025
        )
        
        # Convert back to integer year and re-add the stage column
        stage_imputed['year'] = stage_imputed['year_datetime'].dt.year
        stage_imputed = stage_imputed.drop('year_datetime', axis=1)
        stage_imputed['investment_stage'] = stage
        imputed_stages_list.append(stage_imputed)
    
    # Combine all imputed stage data
    if imputed_stages_list:
        aggregated_funding_stages_df = pd.concat(imputed_stages_list, ignore_index=True)

    # IPOs and acquisitions
    ipos_df = CB.ipos.query("org_id in @matching_ids")
    acquisitions_df = CB.acquisitions.query("acquiree_id in @matching_ids")

    # EXPORT CHART DATA AS CSVs
    # 1. Time series chart data
    ts_chart_data = ts_df[['year', 'raised_amount_gbp_total', 'n_orgs_founded']].copy()
    ts_chart_data['raised_amount_millions'] = ts_chart_data['raised_amount_gbp_total'] 
    ts_chart_data.to_csv(chart_csvs_dir / f"time_series_chart_data_{category_name}.csv", index=False)
    
    # 2. Investment types chart data  
    investment_types_chart_data = aggregated_funding_types_df.copy()
    investment_types_chart_data.to_csv(chart_csvs_dir / f"investment_types_chart_data_{category_name}.csv", index=False)
    
    # 3. Investment stages chart data
    aggregated_funding_stages_df.to_csv(chart_csvs_dir / f"investment_stages_chart_data_{category_name}.csv", index=False)

    # Save the new funding stages data
    aggregated_funding_stages_df.to_csv(produce_stats_dir / f"funding_stages_{category_name}.csv", index=False)

    # Save IPOs and acquisitions to produce_stats directory
    if len(ipos_df) > 0:
        ipos_df.to_csv(produce_stats_dir / f"ipos_{category_name}.csv", index=False)
    if len(acquisitions_df) > 0:
        acquisitions_df.to_csv(produce_stats_dir / f"acquisitions_{category_name}.csv", index=False)
        
    # Fig variables
    prefix = f"{charts_dir}/{category_name}_"
    _scale = 2

    # Investment amounts (total)
    fig = charts.ts_bar(
        ts_df,
        variable='raised_amount_gbp_total',
        variable_title="Raised amount, £ millions",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Funding raised over time for {category_name}")
    fig.save(f"{prefix}raised_amount.png", scale_factor=_scale)

    # Number of companies
    fig = charts.ts_bar(
        ts_df,
        variable='n_orgs_founded',
        variable_title="Number of companies founded",
        category_column="_category",
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of founded {category_name} companies")
    fig.save(f"{prefix}no_of_companies.png", scale_factor=_scale)    

    # Investment amounts (by detailed type)
    investment_types_fig = analysis_crunchbase.chart_investment_types(aggregated_funding_types_df)
    investment_types_fig = charts.configure_plots(
        investment_types_fig, chart_title=f"Breakdown of investment types for {category_name}"
    )
    investment_types_fig.save(f"{prefix}investment_types.png", scale_factor=_scale)

    # Investment counts (by detailed type)
    investment_types_counts_fig = analysis_crunchbase.chart_investment_types_counts(aggregated_funding_types_df)
    investment_types_counts_fig = charts.configure_plots(
        investment_types_counts_fig, chart_title=f"Number of investments by type for {category_name}"
    )
    investment_types_counts_fig.save(f"{prefix}investment_types_counts.png", scale_factor=_scale)

    # NEW: Investment amounts (by funding stage - grouped)
    funding_stages_fig = chart_funding_stages(
        aggregated_funding_stages_df, 
        chart_title=f"Funding amounts by stage for {category_name}"
    )
    funding_stages_fig = charts.configure_plots(funding_stages_fig)
    funding_stages_fig.save(f"{prefix}funding_stages.png", scale_factor=_scale)

    # NEW: Investment counts (by funding stage - grouped)
    funding_stages_counts_fig = chart_funding_stages_counts(
        aggregated_funding_stages_df, 
        chart_title=f"Number of investments by stage for {category_name}"
    )
    funding_stages_counts_fig = charts.configure_plots(funding_stages_counts_fig)
    funding_stages_counts_fig.save(f"{prefix}funding_stages_counts.png", scale_factor=_scale)

    return ts_df, growth_rates_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df

In [9]:
# Run stats using simplified directory structure

config_name = "upfront_costs_innovation"
print(f"Processing {config_name}")

# Load data from the simplified directory structure
relevant_df = pd.read_csv(raw_dir / f"relevant_{primary_config}.csv")
relevant_check_df = pd.read_json(raw_dir / f"llm_check_MS_{primary_config}.jsonl", lines=True)
relevant_checked_df = relevant_df.merge(
    relevant_check_df[['id', 'is_relevant']], on='id', how='left'
)
matching_ids = relevant_checked_df.query("is_relevant == 'yes'").id.tolist()

print(f"Found {len(matching_ids)} relevant companies for {config_name}")

if len(matching_ids) == 0:
    print(f"No relevant companies found for {config_name}, skipping analysis")
else:
    ts_df, growth_rates_df, ipos_df, acquisitions_df, matchings_orgs_df, funding_rounds_df = \
        produce_stats(CB, matching_ids, config_name, OUTPUT_DIR)

    # Save consolidated CSV files to produce_stats directory
    ts_df.assign(theme=config_name).to_csv(produce_stats_dir / "all_ts_df.csv", index=False)
    growth_rates_df.to_csv(produce_stats_dir / "all_growth_rates.csv", index=False)
    ipos_df.assign(theme=config_name).to_csv(produce_stats_dir / "all_ipos_df.csv", index=False)
    acquisitions_df.assign(theme=config_name).to_csv(produce_stats_dir / "all_acquisitions_df.csv", index=False)
    funding_rounds_df.assign(theme=config_name).to_csv(produce_stats_dir / "all_funding_rounds_df.csv", index=False)
    matchings_orgs_df.assign(theme=config_name).to_csv(produce_stats_dir / "all_orgs.csv", index=False)

    # Load and save the funding stages data that was created in produce_stats
    funding_stages_df = pd.read_csv(produce_stats_dir / f"funding_stages_{config_name}.csv")
    funding_stages_df.assign(theme=config_name).to_csv(produce_stats_dir / "all_funding_stages.csv", index=False)

    # LANDSCAPE ANALYSIS - updated with simplified directory structure
    id_condition = "id in ('{}')".format("', '".join(matching_ids))
    vectors_df = CB.VectorDB.vector_db.search().where(id_condition).limit(30000).to_pandas()    

    if len(vectors_df) > 0:  # Only generate landscape if we have data
        try:
            fig, cb_viz_df = viz_landscape.generate_crunchbase_landscape(
                vectors_df, CB, min_cluster_size=min(15, len(vectors_df)//3), n_keyword_clusters=10
            )
            # Save landscape to charts directory
            landscape_path = charts_dir / f"cb_landscape_{config_name}.html"
            fig.save(str(landscape_path))
            print(f"Landscape saved to: {landscape_path}")
            
            # Create export dataframe with landscape data
            export_df = (
                relevant_checked_df
                .merge(
                    cb_viz_df[['id', 'category', 'keyword_cluster', 'recent_funding', 'region']]
                    .rename(columns={"region": "nesta_region"}),
                    on='id', how='left'
                )
            )   
            export_df.to_csv(produce_stats_dir / f"cb_landscape_data_{config_name}.csv", index=False)
            export_df.assign(theme=config_name).to_csv(produce_stats_dir / "all_viz_df.csv", index=False)
            
        except Exception as e:
            print(f"Warning: Landscape generation failed for {config_name}: {e}")
            # Create empty landscape file
            empty_df = pd.DataFrame()
            empty_df.to_csv(produce_stats_dir / "all_viz_df.csv", index=False)
    else:
        print(f"Warning: No vector data found for {config_name}")
        empty_df = pd.DataFrame()
        empty_df.to_csv(produce_stats_dir / "all_viz_df.csv", index=False)

    print(f"Analysis complete for {config_name}")
    print(f"Outputs saved to: {OUTPUT_DIR}")
    print(f"Total companies analyzed: {len(matchings_orgs_df)} relevant companies")

Processing upfront_costs_innovation
Found 7 relevant companies for upfront_costs_innovation
2025-07-30 17:03:45,201 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/funding_rounds_full.parquet
2025-07-30 17:03:54,244 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: data/crunchbase/enriched/funding_rounds_full.parquet
2025-07-30 17:03:54,391 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/Crunchbase_2025-07-28/ipos.parquet
2025-07-30 17:03:55,266 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: data/crunchbase/Crunchbase_2025-07-28/ipos.parquet
2025-07-30 17:03:55,271 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/Crunchbase_2025-07-28/acquisitions.parquet
2025-07-30 17:03:57,267 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: da

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


2025-07-30 17:04:04,533 - root - ERROR - An error occurred while reducing outliers: Found array with 0 sample(s) (shape=(0, 384)) while a minimum of 1 is required by check_pairwise_arrays.
Analysis complete for upfront_costs_innovation
Outputs saved to: /Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/data/upfront_costs_innovation/cb
Total companies analyzed: 7 relevant companies


In [10]:
## Enhanced Country Distribution Analysis

import pandas as pd
import matplotlib.pyplot as plt

# List of EU country codes (ISO Alpha-3)
eu_countries = {
    'AUT', 'BEL', 'BGR', 'HRV', 'CYP', 'CZE', 'DNK', 'EST', 'FIN', 'FRA', 'DEU', 'GRC', 'HUN',
    'IRL', 'ITA', 'LVA', 'LTU', 'LUX', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'ESP', 'SWE'
}

# Function to map country codes
def map_country(code):
    if code == 'GBR':
        return 'UK'
    elif code in eu_countries:
        return 'EU'
    elif pd.isna(code):
        return 'Unknown'
    else:
        return code

# Enhanced geographic analysis using simplified directory structure
config_name = "upfront_costs_innovation"
print(f"=== Enhanced Country Analysis for {config_name} ===")

# Load the filtered companies CSV from simplified directory structure
companies_df = pd.read_csv(filtered_dir / f"relevant_{primary_config}_llm_filtered.csv")

# Load the funding rounds data from produce_stats
funding_rounds_df = pd.read_csv(produce_stats_dir / "all_funding_rounds_df.csv")

# Add country grouping to funding rounds
funding_rounds_df['country_grouped'] = funding_rounds_df['country_code'].apply(map_country)

# Add investment stage mapping
funding_rounds_df['investment_stage'] = funding_rounds_df['investment_type'].map(investment_type_to_stage)

# 1. FUNDING AMOUNTS BY COUNTRY
country_funding = (
    funding_rounds_df.groupby('country_grouped')
    .agg({
        'raised_amount_gbp': 'sum',
        'funding_round_id': 'count'
    })
    .reset_index()
    .rename(columns={
        'raised_amount_gbp': 'total_funding_gbp',
        'funding_round_id': 'num_funding_rounds'
    })
    .sort_values('total_funding_gbp', ascending=False)
)

# Convert to millions for readability
country_funding['total_funding_millions'] = country_funding['total_funding_gbp'] / 1_000_000

# 2. FUNDING BREAKDOWN BY INVESTMENT STAGE PER COUNTRY
country_stage_breakdown = (
    funding_rounds_df.groupby(['country_grouped', 'investment_stage'])
    .agg({
        'raised_amount_gbp': 'sum',
        'funding_round_id': 'count'
    })
    .reset_index()
    .rename(columns={
        'raised_amount_gbp': 'stage_funding_gbp',
        'funding_round_id': 'stage_funding_rounds'
    })
)

# Convert to millions
country_stage_breakdown['stage_funding_millions'] = country_stage_breakdown['stage_funding_gbp'] / 1_000_000

# Save data to CSV (both main data and chart data)
country_funding.to_csv(produce_stats_dir / f"country_funding_{config_name}.csv", index=False)
country_stage_breakdown.to_csv(produce_stats_dir / f"country_stage_breakdown_{config_name}.csv", index=False)

# EXPORT CHART DATA AS CSVs
# Country funding chart data (for bar chart)
country_funding_chart_data = country_funding[['country_grouped', 'total_funding_millions', 'num_funding_rounds']].copy()
country_funding_chart_data.to_csv(chart_csvs_dir / f"country_funding_chart_data_{config_name}.csv", index=False)

# Country stage breakdown chart data (for stacked bar chart)
# Pivot data for easier chart consumption
stage_pivot_chart_data = country_stage_breakdown.pivot(
    index='country_grouped', 
    columns='investment_stage', 
    values='stage_funding_millions'
).fillna(0).reset_index()
stage_pivot_chart_data.to_csv(chart_csvs_dir / f"country_stage_breakdown_chart_data_{config_name}.csv", index=False)

# Also save the original format for detailed analysis
country_stage_breakdown.to_csv(chart_csvs_dir / f"country_stage_breakdown_detailed_{config_name}.csv", index=False)

# Print summary statistics
print(f"\\nFunding Summary for {config_name}:")
print(f"Total funding across all countries: £{country_funding['total_funding_gbp'].sum()/1_000_000:.1f}M")
print(f"Number of countries with funding: {len(country_funding)}")
print("\\nTop 5 countries by funding:")
for _, row in country_funding.head(5).iterrows():
    print(f"  {row['country_grouped']}: £{row['total_funding_millions']:.1f}M ({row['num_funding_rounds']} rounds)")

print("\\nInvestment stage breakdown across all countries:")
stage_totals = country_stage_breakdown.groupby('investment_stage')['stage_funding_gbp'].sum() / 1_000_000
for stage, amount in stage_totals.sort_values(ascending=False).items():
    print(f"  {stage}: £{amount:.1f}M")

print(f"\\n📄 Chart data exported to: {chart_csvs_dir}")
print("=" * 60)

=== Enhanced Country Analysis for upfront_costs_innovation ===
\nFunding Summary for upfront_costs_innovation:
Total funding across all countries: £0.0M
Number of countries with funding: 2
\nTop 5 countries by funding:
  EU: £0.0M (5 rounds)
  UK: £0.0M (3 rounds)
\nInvestment stage breakdown across all countries:
  other: £0.0M
  early_stage: £0.0M
\n📄 Chart data exported to: /Users/aidan.kelly/nesta/discovery/discovery_mission_radar/discovery_mission_radar/data/upfront_costs_innovation/cb/chart_csvs


In [12]:
# Google Sheets Upload Configuration - MAIN DATA
from discovery_utils.utils import google
from discovery_utils.getters import crunchbase

# Set your Google Sheet ID here
sheet_id = "1Hfss-nGAITYCrcKUoDiV26WGEkhBFGtMXLT2i-axaVQ"  # Replace with your Upfront Costs Innovation Google Sheet ID

# Optionally specify a single sheet to update (e.g. "llm_filtered_companies"); set to None to update all MAIN DATA
SHEET_TO_UPDATE = None  # Set to specific sheet name to update only that sheet

# Process and upload MAIN DATA using simplified directory structure
config_name = "upfront_costs_innovation"
print(f"Processing {config_name} for Google Sheets upload (MAIN DATA ONLY)...")

try:
    # Read the main data files from produce_stats directory
    all_ts_df = pd.read_csv(produce_stats_dir / "all_ts_df.csv")
    all_growth_rates = pd.read_csv(produce_stats_dir / "all_growth_rates.csv") 
    all_ipos_df = pd.read_csv(produce_stats_dir / "all_ipos_df.csv")
    all_acquisitions_df = pd.read_csv(produce_stats_dir / "all_acquisitions_df.csv")
    all_funding_rounds_df = pd.read_csv(produce_stats_dir / "all_funding_rounds_df.csv")
    all_orgs = pd.read_csv(produce_stats_dir / "all_orgs.csv")
    
    # Load relevant data from raw and filtered directories to get ai_relevance_check
    relevant_df = pd.read_csv(raw_dir / f"relevant_{primary_config}.csv")
    relevant_check_df = pd.read_json(raw_dir / f"llm_check_MS_{primary_config}.jsonl", lines=True)

    # Merge to get the ai_relevance_check column
    relevant_checked_df = relevant_df.merge(
        relevant_check_df[['id', 'is_relevant']], on='id', how='left'
    )
    
    # Process funding rounds data similar to reference code
    _all_funding_rounds_df = (
        all_funding_rounds_df
        .assign(investment_stage = lambda df: df.investment_type.map(crunchbase.investment_type_to_stage()))
        .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
    )[cols_funding_rounds]
    
    # Process organizations data - properly handle the merge and rename
    print(f"  - Raw data columns: {len(relevant_checked_df.columns)} columns")
    
    # Create the processed organizations dataframe with proper renaming
    _all_orgs = (
        relevant_checked_df
        .rename(columns={
            'category': 'landscape_category', 
            'keyword_cluster': 'landscape_keyword_cluster', 
            "is_relevant": "ai_relevance_check"  # This is the key rename
        })
        .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
        .assign(theme=config_name)  # Add theme column
    )
    
    print(f"  - After processing: {len(_all_orgs.columns)} columns")
    print(f"  - Has ai_relevance_check: {'ai_relevance_check' in _all_orgs.columns}")
    print(f"  - Has theme: {'theme' in _all_orgs.columns}")
    
    # Only select columns that actually exist in the dataframe
    available_cols = [col for col in cols_companies if col in _all_orgs.columns]
    missing_cols = [col for col in cols_companies if col not in _all_orgs.columns]
    
    if missing_cols:
        print(f"  - Warning: Missing columns will be skipped: {missing_cols}")
    
    print(f"  - Will select {len(available_cols)} columns out of {len(cols_companies)} requested")
    
    _all_orgs = _all_orgs[available_cols]
    
    # ALSO export only the LLM-approved rows
    _llm_only = _all_orgs.query("ai_relevance_check == 'yes'").copy()
    
    print(f"  - Companies: {len(_all_orgs)} (pre-LLM), {len(_llm_only)} (LLM-approved)")
    print(f"Data prepared for {config_name}:")
    print(f"  - Companies: {len(_all_orgs)}")
    print(f"  - Funding rounds: {len(_all_funding_rounds_df)}")
    print(f"  - IPOs: {len(all_ipos_df)}")
    print(f"  - Acquisitions: {len(all_acquisitions_df)}")
    
    # Upload MAIN DATA ONLY to Google Sheets 
    sheet_names = {
        "crunchbase_companies": _all_orgs,
        "llm_filtered_companies": _llm_only,
        "crunchbase_funding": _all_funding_rounds_df,
        "crunchbase_acquisitions": all_acquisitions_df,
        "crunchbase_ipos": all_ipos_df,
        "crunchbase_timeseries": all_ts_df,
        "crunchbase_growth_rates": all_growth_rates,
    }
    
    # Only upload non-empty dataframes (and optionally limit to one sheet)
    if SHEET_TO_UPDATE:
        upload_data = {name: df for name, df in sheet_names.items()
                       if name == SHEET_TO_UPDATE and not df.empty}
    else:
        upload_data = {name: df for name, df in sheet_names.items() 
                      if not df.empty and len(df.columns) > 0}
    
    print(f"  - Preparing to upload {len(upload_data)} MAIN DATA sheets")
    
    if upload_data:
        google.upload_data_to_gsheet(sheet_id, upload_data)
        print(f"✓ Uploaded {len(upload_data)} MAIN DATA sheets for {config_name} to Google Sheet: {sheet_id}")
        
        # Apply formatting to key sheets with error handling
        try:
            google.format_gsheet(sheet_id, "crunchbase_companies", freeze_cols=4)
            google.format_gsheet(sheet_id, "crunchbase_funding", freeze_cols=2)
                    
        except Exception as e:
            print(f"  - Warning: Could not apply formatting: {e}")
        
    else:
        print(f"⚠️  No data to upload for {config_name}")
        
except FileNotFoundError as e:
    print(f"⚠️  Could not find data files for {config_name}: {e}")
    print("Make sure you've run the analysis sections above first.")
except Exception as e:
    print(f"❌ Error uploading {config_name} to Google Sheets: {e}")
    import traceback
    traceback.print_exc()

print("\\n🎯 Main data Google Sheets upload complete!")

Processing upfront_costs_innovation for Google Sheets upload (MAIN DATA ONLY)...
  - Raw data columns: 63 columns
  - After processing: 65 columns
  - Has ai_relevance_check: True
  - Has theme: True
  - Warning: Missing columns will be skipped: ['landscape_category', 'landscape_keyword_cluster', 'recent_funding']
  - Will select 37 columns out of 40 requested
  - Companies: 526 (pre-LLM), 7 (LLM-approved)
Data prepared for upfront_costs_innovation:
  - Companies: 526
  - Funding rounds: 8
  - IPOs: 1
  - Acquisitions: 0
  - Preparing to upload 6 MAIN DATA sheets
2025-07-30 17:06:45,669 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:06:48,398 - root - INFO - Uploading DataFrame to sheet: crunchbase_companies
2025-07-30 17:07:08,372 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:07:11,646 - root - INFO - Uploading DataFrame to sheet: llm_filtered_companies
2025-07-30 17:07:26,347 - root - INFO - Connected to Google Sheet: LCH innovation


In [13]:
# Google Sheets Upload - CHART DATA ONLY
# Run this cell separately to upload just the chart data when needed

# Use the same sheet ID as above
chart_sheet_id = "1Hfss-nGAITYCrcKUoDiV26WGEkhBFGtMXLT2i-axaVQ"  # Same ID as main data

# Optionally specify a single chart to update, or None for all charts
CHART_TO_UPDATE = None  # e.g. "chart_timeseries" or None for all

print("🔥 Starting CHART DATA upload...")

# Process and upload CHART DATA using simplified directory structure
config_name = "upfront_costs_innovation"
print(f"Processing {config_name} for Chart Data upload...")

try:
    chart_data = {}
    chart_files = {
        "chart_timeseries": f"time_series_chart_data_{config_name}.csv",
        "chart_investment_types": f"investment_types_chart_data_{config_name}.csv", 
        "chart_investment_stages": f"investment_stages_chart_data_{config_name}.csv",
        "chart_country_funding": f"country_funding_chart_data_{config_name}.csv",
        "chart_country_stages": f"country_stage_breakdown_chart_data_{config_name}.csv"
    }
    
    # Load chart data with error handling
    for tab_name, filename in chart_files.items():
        chart_file_path = chart_csvs_dir / filename
        if chart_file_path.exists():
            try:
                chart_df = pd.read_csv(chart_file_path)
                if not chart_df.empty:
                    chart_data[tab_name] = chart_df
                    print(f"  ✓ Loaded {tab_name}: {len(chart_df)} rows")
                else:
                    print(f"  ⚠️  Chart file is empty: {filename}")
            except Exception as e:
                print(f"  ❌ Error loading chart file {filename}: {e}")
        else:
            print(f"  ⚠️  Chart file not found: {filename}")
    
    # Filter to specific chart if requested
    if CHART_TO_UPDATE:
        if CHART_TO_UPDATE in chart_data:
            chart_data = {CHART_TO_UPDATE: chart_data[CHART_TO_UPDATE]}
            print(f"  - Uploading only: {CHART_TO_UPDATE}")
        else:
            print(f"  ⚠️  Requested chart {CHART_TO_UPDATE} not found")
    
    print(f"  - Chart datasets ready: {len(chart_data)}")
    
    if chart_data:
        # Upload chart data to Google Sheets
        google.upload_data_to_gsheet(chart_sheet_id, chart_data)
        print(f"✓ Uploaded {len(chart_data)} CHART sheets for {config_name} to Google Sheet: {chart_sheet_id}")
        
        # Apply formatting to chart sheets
        try:
            for chart_tab in chart_data.keys():
                google.format_gsheet(chart_sheet_id, chart_tab, freeze_cols=1)
            print(f"  ✓ Applied formatting to {len(chart_data)} chart sheets")
                    
        except Exception as e:
            print(f"  - Warning: Could not apply formatting to charts: {e}")
        
    else:
        print(f"⚠️  No chart data to upload for {config_name}")
        
except Exception as e:
    print(f"❌ Error uploading chart data for {config_name}: {e}")
    import traceback
    traceback.print_exc()

print("\\n🎯 Chart data Google Sheets upload complete!")

🔥 Starting CHART DATA upload...
Processing upfront_costs_innovation for Chart Data upload...
  ✓ Loaded chart_timeseries: 12 rows
  ✓ Loaded chart_investment_types: 6 rows
  ✓ Loaded chart_investment_stages: 24 rows
  ✓ Loaded chart_country_funding: 2 rows
  ✓ Loaded chart_country_stages: 2 rows
  - Chart datasets ready: 5
2025-07-30 17:19:33,315 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:19:36,988 - root - INFO - Uploading DataFrame to sheet: chart_timeseries
2025-07-30 17:19:47,723 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:19:50,231 - root - INFO - Uploading DataFrame to sheet: chart_investment_types
2025-07-30 17:20:02,443 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:20:05,191 - root - INFO - Uploading DataFrame to sheet: chart_investment_stages
2025-07-30 17:20:17,402 - root - INFO - Connected to Google Sheet: LCH innovation
2025-07-30 17:20:19,500 - root - INFO - Uploading DataFrame to sheet: cha

## 🚀 Usage Instructions

**Quick Start**: Run all cells sequentially. If filtered data exists, LLM processing will be skipped.

**Key Parameters to Modify**:
```python
# Adjust filtering sensitivity
SCORE_THRESHOLD = 0.2  # Lower = more results, 0.4 = more selective

# Focus on specific funding stages
INCLUDE_LATE_STAGE = False  # Exclude late-stage funding
INCLUDE_GROWTH_STAGE = False  # Focus only on early-stage
INCLUDE_OTHER_STAGE = False  # Exclude other funding types

# Limit Google Sheets upload
SHEET_TO_UPDATE = "crunchbase_companies"  # Upload specific sheet only
CHART_TO_UPDATE = "chart_timeseries"  # Upload specific chart only
```

**Output Structure**:
- **Raw/Filtered**: `data/upfront_costs_innovation/cb/raw/` and `filtered/`
- **Analysis**: `data/upfront_costs_innovation/cb/produce_stats/` - charts, CSVs, landscape
- **Chart Data**: `data/upfront_costs_innovation/cb/chart_csvs/` - ready for visualization
- **Google Sheets**: Automatically uploaded with formatting applied

**Before Running**:
- Ensure `config_upfront_costs.yaml` exists in notebook directory
- Update Google Sheet ID in upload sections if needed
- Verify sufficient disk space (~100MB for outputs)

---

## 🔧 Troubleshooting

**Expected Results**:
- Upfront Costs Innovation Companies: 20-40 companies
- Focus on cost reduction, installation efficiency, and financing innovation
- Analysis of commercial investment patterns in heat pump cost reduction

**Common Issues**:
1. **Missing Configuration File**:
   ```
   Error: FileNotFoundError: config_upfront_costs.yaml
   Solution: Ensure YAML config file exists in notebook directory
   ```

2. **LLM Processing Timeouts**:
   ```
   Error: LLM batch processing timeout
   Solution: Reduce batch_size from 15 to 10, increase sleep_time to 1.0
   ```

3. **Google Sheets Upload Failures**:
   ```
   Error: 403 Insufficient permissions
   Solution: Verify service account has edit access to target sheet
   ```

4. **Landscape Generation Errors**:
   ```
   Error: Found array with 0 sample(s)
   Solution: Check filtered data has sufficient companies (>10)
   ```

**Data Quality Validation**:
- If results are significantly outside expected ranges, check:
  - Search term configuration in YAML file
  - SCORE_THRESHOLD setting (try 0.2-0.4 range)
  - LLM relevance checking logic

**Performance Tips**:
- **For development**: Set `SHEET_TO_UPDATE` and `CHART_TO_UPDATE` to process single sheets
- **For memory**: Clear large variables between sections if needed
- **For speed**: Use existing filtered data when possible
- **For debugging**: Check intermediate CSV files in raw/ and filtered/ directories

---